# Mervin/Mervis -- Gemma 4 E2B LoRA fine-tune (Google Colab)

LoRA fine-tunes **Gemma 4 E2B-it** (the smaller, faster sibling of the arena's
Gemma 4 E4B) on the Mervin/Mervis persona dataset with
[Unsloth](https://github.com/unslothai/unsloth), exports a 4-bit **Q4_K_M GGUF**
(~3.4 GB), and uploads it to Hugging Face -- where `serve.py` auto-downloads it.

**No system prompt.** The Mervin/Mervis tags and behavior come *purely* from
fine-tuning: training data is bare `user -> assistant`, so the model produces
the format with or without any system prompt (the direction we want for every
arena model). Verified on an A100.

Trains on Google Colab. A GPU is required -- a free T4 is plenty for E2B with
Unsloth 4-bit; A100/L4 are faster.

### Before you run
- Runtime -> Change runtime type -> **GPU**.
- Add a Colab **secret** `HF_TOKEN` (key icon) with a Hugging Face **write**
  token. The upload cell reads it via `userdata`; it is never written into the
  notebook.

### Two Gemma 4 gotchas (handled below)
- **transformers version:** Gemma 4 uses `model_type "gemma4"`, which the 4.56.2
  used by older notebooks does NOT recognize. We install a newer transformers
  (`>4.56.2,<=5.5.0`, the range Unsloth still supports).
- **PEFT:** Gemma 4 wraps projections in `Gemma4ClippableLinear`; we pin
  `peft>=0.19.0` so PEFT recognizes them.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

In [ ]:
# Unsloth + a dependency set it supports. Gemma 4 needs a newer transformers than
# 4.56.2 (gemma4 arch); bump to the newest Unsloth still supports (<=5.5.0).
# peft>=0.19.0 is required for Gemma 4's Gemma4ClippableLinear. Drop the runtime's
# broken torchao (optional for bnb 4-bit). We read versions via importlib.metadata
# so we do NOT import transformers here (a clean run then loads the new one).
!pip install --upgrade -q unsloth unsloth_zoo
!pip install -q "transformers>4.56.2,<=5.5.0,!=5.0.0,!=5.1.0" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0,!=0.19.0" "peft>=0.19.0"
!pip uninstall -q -y torchao
import importlib.metadata as m
print("transformers", m.version("transformers"), "| trl", m.version("trl"),
      "| datasets", m.version("datasets"), "| peft", m.version("peft"))
# If you ran a cell that imported transformers BEFORE this one, do
# Runtime -> Restart session, then run from the next cell down.

In [ ]:
import os
# Some Colab torch images have a broken torch.compile/inductor that crashes
# Unsloth training/inference. Run eager (slower but correct) to be safe.
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/gemma-4-E2B-it",   # not gated; google/gemma-4-E2B-it also works
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = True,
    full_finetuning = False,
)
print("loaded gemma-4-E2B-it")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 32,
    lora_dropout               = 0.05,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)
model.print_trainable_parameters()

## Dataset -- no system prompt

Each example is the full `user -> assistant` turn rendered with Gemma 4's native
chat template (which uses `<|turn>...<turn|>` markers). We render the **whole
turn** rather than appending `eos_token`, so the model-turn terminator is the
correct `<turn|>` token (Gemma 4's eos is `<eos>`, which is NOT the turn marker).
The persona is carried entirely by the assistant targets.

In [ ]:
import csv, io, urllib.request
from datasets import Dataset

CSV_URL = "https://raw.githubusercontent.com/CarlFreeAiEngineer/merv/main/mervin_mervis_finetune.csv"
raw  = urllib.request.urlopen(CSV_URL).read().decode("utf-8")
rows = list(csv.DictReader(io.StringIO(raw)))
print(f"Loaded {len(rows)} rows")

# Rows are 1-, 2-, or 3-turn (prompt/response, +prompt2/response2, +prompt3/response3).
# Render the WHOLE conversation so the model keeps the Mervin/Mervis format on
# FOLLOW-UP turns, not just turn 1.
def to_text(r):
    msgs = [{"role": "user", "content": r["prompt"]},
            {"role": "assistant", "content": r["response"]}]
    for i in ("2", "3"):
        p, a = (r.get("prompt" + i) or "").strip(), (r.get("response" + i) or "").strip()
        if p and a:
            msgs += [{"role": "user", "content": p}, {"role": "assistant", "content": a}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

ds = Dataset.from_list([to_text(r) for r in rows])
n2 = sum(1 for r in rows if r.get("prompt2")); n3 = sum(1 for r in rows if r.get("prompt3"))
print(f"{len(rows)-n2} single, {n2-n3} two-turn, {n3} three-turn")
print(ds[0]["text"][:500])


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 6,   # small model needs more passes for a reliable format
        learning_rate               = 2e-4,
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        logging_steps               = 5,
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)
trainer.train()

## Sanity check -- bare user message, no system prompt

Gemma 4's `tokenizer` is a multimodal *processor*; for plain text we render the
template to a string and tokenize with its inner text tokenizer.

In [ ]:
# Both-tags check -- runs BEFORE export.
#   single + 2nd-turn = HARD GATE (raises -> a "Run all" STOPS here if a tag is dropped)
#   3rd-turn          = DIAGNOSTIC ONLY (reported, never blocks; longer context is best-effort)
import re
FastLanguageModel.for_inference(model)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)

def ask_msgs(msgs):
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = text_tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)
    return text_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def both_tags(t):
    return bool(re.search(r"<Mervin>.*?</Mervin>", t, re.S) and re.search(r"<Mervis>.*?</Mervis>", t, re.S))

def convo(turns):
    msgs, reply = [], ""
    for u in turns:
        msgs.append({"role": "user", "content": u})
        reply = ask_msgs(msgs)
        msgs.append({"role": "assistant", "content": reply})
    return reply

SINGLE = ["What is 2+2?", "Tell me about Mondays.", "Recommend a book.", "Explain gravity."]
PAIRS = [("What do you think about Mondays?", "And what about Fridays?"),
         ("Should I get a cat?", "What about a dog instead?"),
         ("Is it going to rain tomorrow?", "And the day after?"),
         ("Tell me about the ocean.", "What lives in it?"),
         ("What is gravity?", "Does it ever take a day off?"),
         ("Recommend a movie.", "What about a cheerful one?")]
TRIPLES = [("What is 2+2?", "And times ten?", "Now minus three?"),
           ("Tell me about dogs.", "What about cats?", "And goldfish?"),
           ("Is coffee any good?", "What about tea?", "And plain water?"),
           ("Name a planet.", "Another one?", "The smallest?")]

t1 = t2 = t3 = 0
for q in SINGLE:
    r = convo([q]); g = both_tags(r); t1 += g
    print(("OK " if g else "BAD"), "T1 |", q, "->", r[:62])
for q1, q2 in PAIRS:
    r = convo([q1, q2]); g = both_tags(r); t2 += g
    print(("OK " if g else "BAD"), "T2 |", q2, "->", r[:62])
for q1, q2, q3 in TRIPLES:
    r = convo([q1, q2, q3]); g = both_tags(r); t3 += g
    print(("OK " if g else "BAD"), "T3 |", q3, "->", r[:62])

print(f"\nsingle {t1}/{len(SINGLE)}   2nd-turn {t2}/{len(PAIRS)} [GATE]   3rd-turn {t3}/{len(TRIPLES)} [diagnostic]")
assert t1 == len(SINGLE), f"single-turn regression: {t1}/{len(SINGLE)}"
assert t2 >= len(PAIRS) - 1, f"2ND-TURN REGRESSION: only {t2}/{len(PAIRS)} follow-up replies kept both tags. Do NOT upload."
print(f"GATE PASSED -- safe to export/upload. (3rd-turn {t3}/{len(TRIPLES)} informational only.)")


## Export Q4_K_M GGUF

Gemma 4 is multimodal, so the converter emits a text `Q4_K_M.gguf` (~3.4 GB)
plus a separate `*-mmproj` vision projector. We do text chat, so we upload only
the text GGUF.

In [ ]:
import glob, os, time
t0 = time.time()
model.save_pretrained_gguf("gemma-merv-gguf", tokenizer, quantization_method="q4_k_m")
print("export done in", round(time.time()-t0), "s")
for f in sorted(glob.glob("/content/**/gemma*Q4_K_M.gguf", recursive=True)):
    print(round(os.path.getsize(f)/1e9, 2), "GB ", f)

## Upload directly to Hugging Face

Pushes the text GGUF to `freeideas/merv-gemma4e2b` as `model-q4_k_m.gguf`, where
`serve.py` auto-downloads it. Token comes from the Colab `HF_TOKEN` secret.

In [ ]:
import glob
from google.colab import userdata
from huggingface_hub import HfApi

tok  = userdata.get("HF_TOKEN")
repo = "freeideas/merv-gemma4e2b"
src  = glob.glob("/content/**/gemma*Q4_K_M.gguf", recursive=True)[0]   # text model only

api = HfApi()
print("uploading as model-q4_k_m.gguf ->", repo, "( as", api.whoami(token=tok)["name"], ")")
api.create_repo(repo, repo_type="model", exist_ok=True, token=tok)
api.upload_file(path_or_fileobj=src, path_in_repo="model-q4_k_m.gguf",
                repo_id=repo, repo_type="model", token=tok)
print("done -> https://huggingface.co/" + repo)

## In the arena

`serve.py` and `index.html` carry a `gemma2b` entry, so any host running
`serve.py` auto-downloads `model-q4_k_m.gguf` from `freeideas/merv-gemma4e2b` on
startup and the model appears in the dropdown. See this folder's `README.md`.